In [1]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device(
    'cuda' if torch.cuda.is_available() else # NVIDIA
    'mps' if torch.backends.mps.is_available() else # MAC
    'cpu' 
)

print(f'현재 사용중인 연산 디바이스 : {device}')

현재 사용중인 연산 디바이스 : mps


In [2]:
raw_train_data = datasets.FashionMNIST(
    root='data', train=True, download=True, transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root='data', train=False, download=True, transform=ToTensor()
)

In [ ]:


train_size = 50000
val_size = 10000

train_data, val_data = random_split(
    raw_train_data, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)


print(f'훈련 셋 : {len(train_data)}')
print(f'검증 셋 : {len(val_data)}')
print(f'테스트 셋 : {len(test_data)}')

훈련 셋 : 50000
검증 셋 : 10000
테스트 셋 : 10000


In [4]:
import torch.nn as nn
import torch.optim as optim

class AdvancedFashionClassifier(nn.Module):
    # 사용할 층(Layer)을 정의
    def __init__(self):
        super().__init__()

        # (28, 28) -> 특성은 1차원으로 변경 (784,)
        # nn.Flatten() -> 2차원 텐서 -> 1차원 텐서 - 입력층
        self.flatten = nn.Flatten()

        # 은닉층 ( 784 -> 64 )
        self.hidden_layer = nn.Linear(784, 64)

        # 활성화 함수 (ReLU)
        self.relu = nn.ReLU()

        # Dropout - 0 ~ 1, 0.2 ~ 0.5
        self.dropout = nn.Dropout(p=0.2)

        # 출력층 128 -> 10
        self.output_layer = nn.Linear(64, 10)

    # 순전파(Feed Forward)
    def forward(self, x):
        out = self.flatten(x) # 28 X 28 -> 784 # 입력층 처리
        out = self.hidden_layer(out) # 784 -> 64 # 은닉층 처리
        out = self.relu(out) # 활성화 함수 통과 (선형적 -> 비선형적)
        out = self.dropout(out) # 훈련시에만 필요, 과대적합 방지 
        out = self.output_layer(out) # 64 -> 10 출력층

        return out


In [5]:
model = AdvancedFashionClassifier().to(device) # device에 지정된 기기로 연결

total_params = sum(p.numel() for p in  model.parameters() if p.requires_grad)
print(f'학습 가능한 총 모델 파라미터 개수 : {total_params:,}개')

학습 가능한 총 모델 파라미터 개수 : 50,890개


In [6]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [7]:
# 손실 함수
criterion = nn.CrossEntropyLoss()

# 옵티마이저
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [8]:
epochs = 5 # 최대 에포크
patience_count = 0
best_val_loss = float('inf')

train_losses = []
val_losses = []

for epoch in range(epochs):
    running_loss = 0.0 # 한 에포크(각 배치의 모든 손실)의 손실
    model.train() # 학습 모드 ( 학습에 필요한 것들 활성 - ex) Dropout )

    for images, labels in train_loader: # 미니배치 (64개씩)
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() # 이전 기울기 초기화

        # 순방향 예측
        outputs = model(images) # 순방향

        # 손실 계산
        loss = criterion(outputs, labels)

        # 오차 역전파 수행 -> 기울기 구하기
        loss.backward()

        # 옵티마이저를 통한 가중치 업데이트
        optimizer.step()

        running_loss += loss.item() # 배치마다의 로스를 기록 (1에포크)

    epoch_loss = running_loss / len(train_loader) # 1에포크당 로스 (배치 로스의 평균)
    train_losses.append(epoch_loss)

    # 검증 모드
    model.eval() # 평가시에는 학습시에 필요한 부분 배제 -> [ Dropout, 기울기 업데이트 ]
    val_loss = 0.0

    with torch.no_grad(): # 검증시에는 기울기를 구하지 않음, 학습 관련 연산 하지 않음
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            # 검증 예측 ( 순전파 )
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    epoch_val_loss = val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)


    print(f'Epoch : {epoch + 1}/{epochs} | 훈련 loss : {epoch_loss:.4f}, 검증 로스 : {val_loss:.4f}')

Epoch : 1/5 | 훈련 loss : 1.2950, 검증 로스 : 129.5317
Epoch : 2/5 | 훈련 loss : 0.7827, 검증 로스 : 105.8782
Epoch : 3/5 | 훈련 loss : 0.6761, 검증 로스 : 94.7291
Epoch : 4/5 | 훈련 loss : 0.6198, 검증 로스 : 88.6358
Epoch : 5/5 | 훈련 loss : 0.5778, 검증 로스 : 83.5996
